In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from fredapi import Fred
import yfinance as yf

## Data Collection

Data for this project will be obtained from FRED (Federal Reserve Economic Data) and Yahoo Finance APIs. The following macroeconomic and financial indicators will be examined:

### FRED Indicators
- **Unemployment Rate (UNRATE)** - monthly measure of the percentage of the labour force that is unemployed, a key lagging indicator of economic health
- **Consumer Price Index (CPIAUCSL)** - measures inflation through changes in the price of a basket of consumer goods and services
- **Federal Funds Rate (FEDFUNDS)** - the interest rate at which banks lend to each other overnight, a primary tool of US monetary policy
- **Industrial Production (INDPRO)** - measures real output of the manufacturing, mining and utilities sectors, a leading indicator of economic activity
- **House Prices (CSUSHPISA)** - S&P CoreLogic Case-Shiller US National Home Price Index, captures housing market conditions
- **Yield Curve (T10Y2Y)** - spread between 10-year and 2-year Treasury yields; inversion has preceded every US recession since the 1950s
- **Initial Jobless Claims (ICSA)** - weekly count of new unemployment insurance claims, one of the most timely labour market indicators available
- **Credit Spread (BAA10Y)** - spread between Moody's BAA corporate bond yield and 10-year Treasury yield; widens as default risk rises ahead of recessions
- **Consumer Credit (TOTALSL)** - total outstanding consumer credit, captures household borrowing trends
- **Leading Economic Index (USSLIND)** - OECD composite leading indicator designed to anticipate turning points in economic activity
- **Housing Starts (HOUST)** - number of new residential construction projects begun each month, a forward-looking indicator of economic activity
- **Recession Indicator (USREC)** - NBER defined binary indicator, 1 during recession months and 0 otherwise; used as the target variable

### Yahoo Finance Indicators
- **S&P 500 (^GSPC)** - broad measure of US equity market performance, reflects investor expectations about future economic conditions
- **Gold (GC=F)** - commodity price commonly associated with safe haven demand during periods of economic uncertainty
- **VIX (^VIX)** - CBOE Volatility Index, measures expected market volatility and is commonly referred to as the market fear gauge

In [3]:
# FRED API Connection

FRED_API_KEY = "99a4258f3c059800db6f3473af4d2edf"
fred = Fred(api_key=FRED_API_KEY)

In [4]:
# Start & End Dates for Project

START = "1960-01-01"
END   = pd.Timestamp.today().strftime("%Y-%m-%d")

In [5]:
# FRED Indicators 

unemployment          = fred.get_series("UNRATE",   START, END)
cpi                   = fred.get_series("CPIAUCSL",  START, END)
fed_funds             = fred.get_series("FEDFUNDS",  START, END)
industrial_production = fred.get_series("INDPRO",   START, END)
house_prices          = fred.get_series("CSUSHPISA", START, END)
recession             = fred.get_series("USREC",     START, END)
yield_curve           = fred.get_series("T10Y2Y",    START, END)
jobless_claims        = fred.get_series("ICSA",      START, END)
credit_spread         = fred.get_series("BAA10Y",    START, END)
consumer_credit       = fred.get_series("TOTALSL",   START, END)
lei                   = fred.get_series("USSLIND",   START, END)
housing_starts        = fred.get_series("HOUST",     START, END)

print("FRED data collected")

FRED data collected


In [6]:
# Yahoo Finance Indicators 

sp500_raw  = yf.download("^GSPC",  start=START, end=END, interval="1mo", progress=False)
gold_raw   = yf.download("GC=F",   start=START, end=END, interval="1mo", progress=False)
vix_raw    = yf.download("^VIX",   start=START, end=END, interval="1mo", progress=False)

sp500  = sp500_raw["Close"].squeeze()
gold   = gold_raw["Close"].squeeze()
vix    = vix_raw["Close"].squeeze()

print("Yahoo Finance data collected")

Yahoo Finance data collected


In [7]:
# Align to Monthly Frequency 
# resample all series to month start, forward fill where needed

def to_monthly(series, label):
    s = series.copy()
    s.index = pd.to_datetime(s.index)
    s = s.resample("MS").mean()
    s.name = label
    return s

unemployment          = to_monthly(unemployment,          "unemployment")
cpi                   = to_monthly(cpi,                   "cpi")
fed_funds             = to_monthly(fed_funds,             "fed_funds")
industrial_production = to_monthly(industrial_production, "industrial_production")
house_prices          = to_monthly(house_prices,          "house_prices")
recession             = to_monthly(recession,             "recession")
yield_curve           = to_monthly(yield_curve,           "yield_curve")
jobless_claims        = to_monthly(jobless_claims,        "jobless_claims")
credit_spread         = to_monthly(credit_spread,         "credit_spread")
consumer_credit       = to_monthly(consumer_credit,       "consumer_credit")
lei                   = to_monthly(lei,                   "lei")
housing_starts        = to_monthly(housing_starts,        "housing_starts")
sp500                 = to_monthly(sp500,                 "sp500")
gold                  = to_monthly(gold,                  "gold")
vix                   = to_monthly(vix,                   "vix")

print("All series aligned to monthly frequency")

All series aligned to monthly frequency


In [28]:
# Data Coverage Summary 

series_dict = {
    "unemployment":           unemployment,
    "cpi":                    cpi,
    "fed_funds":              fed_funds,
    "industrial_production":  industrial_production,
    "house_prices":           house_prices,
    "recession":              recession,
    "yield_curve":            yield_curve,
    "jobless_claims":         jobless_claims,
    "credit_spread":          credit_spread,
    "consumer_credit":        consumer_credit,
    "lei":                    lei,
    "housing_starts":         housing_starts,
    "sp500":                  sp500,
    "gold":                   gold,
    "vix":                    vix,
}

print(f"{'Indicator':<25} {'Start':<15} {'End':<15} {'Rows':<10} {'NaNs'}")
print("-" * 75)
for name, series in series_dict.items():
    clean = series.dropna()
    nans  = series.isnull().sum()
    print(f"{name:<25} {str(clean.index.min().date()):<15} {str(clean.index.max().date()):<15} {len(clean):<10} {nans}")

Indicator                 Start           End             Rows       NaNs
---------------------------------------------------------------------------
unemployment              1960-01-01      2026-05-01      796        1
cpi                       1960-01-01      2026-05-01      796        1
fed_funds                 1960-01-01      2026-05-01      797        0
industrial_production     1960-01-01      2026-05-01      797        0
house_prices              1987-01-01      2026-03-01      471        144
recession                 1960-01-01      2026-05-01      797        0
yield_curve               1976-06-01      2026-06-01      601        0
jobless_claims            1967-01-01      2026-06-01      714        0
credit_spread             1986-01-01      2026-06-01      486        0
consumer_credit           1960-01-01      2026-04-01      796        0
lei                       1982-01-01      2020-02-01      458        0
housing_starts            1960-01-01      2026-05-01      797      